In [26]:
# ============================================
# 1. Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')


# ============================================
# 2. Clone GitHub Repository
# ============================================
!git clone https://github.com/Dipgi/lightgcn-res.git


# ============================================
# 3. Move into project folder
# ============================================
%cd lightgcn-res

# ============================================
# Create __init__.py for 'src' to be a package
# ============================================
!touch src/__init__.py

# ============================================
# 4. Install dependencies
# ============================================
!pip install -r requirements.txt


# ============================================
# 5. Fix Python import paths
# ============================================
import sys
sys.path.append('/content/lightgcn-res')


# ============================================
# 6. Verify project structure
# ============================================
!ls

# Data path
DATA_PATH = "/content/drive/MyDrive/lightgcn-res/data/movielens/latest_small/ratings.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'lightgcn-res'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 83 (delta 30), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 65.11 KiB | 1.63 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/lightgcn-res/lightgcn-res/lightgcn-res/lightgcn-res/lightgcn-res
configs  LICENSE    README.md	      results  src
data	 notebooks  requirements.txt  scripts


In [18]:
# Install Requirements

#!pip install torch pandas numpy scipy scikit-learn pyyaml

In [ ]:
# Mount Google Drive

#from google.colab import drive
#drive.mount('/content/drive')

#DATA_PATH = "/content/drive/MyDrive/Light_CGN_project/data/movielens/latest_small/ratings.csv"

Mounted at /content/drive


In [27]:
# Import Libraries

import sys
sys.path.append('/content/lightgcn-res')

import torch
import pandas as pd
import numpy as np
import yaml

from src.model import LightGCN
from src.dataset import load_movielens
from src.metrics import precision_at_k

In [28]:
# Load Configuration

with open("../configs/lightgcn.yaml") as f:
    config = yaml.safe_load(f)

print(config)

{'model': {'model_type': 'lightgcn', 'embed_size': 64, 'n_layers': 3}, 'train': {'batch_size': 1024, 'decay': 0.0001, 'epochs': 1000, 'learning_rate': 0.001, 'eval_epoch': -1, 'top_k': 20}, 'info': {'save_model': False, 'save_epoch': 100, 'metrics': ['recall', 'ndcg', 'precision', 'map'], 'MODEL_DIR': './tests/resources/deeprec/lightgcn/model/lightgcn_model/'}}


In [37]:
# Load Dataset

# Load the ratings data directly using pandas
ratings = pd.read_csv(DATA_PATH)

# Rename 'movieId' column to 'itemId' if it exists, as expected by the model
# This is done here explicitly to ensure 'itemId' is present before further processing.
if 'movieId' in ratings.columns and 'itemId' not in ratings.columns:
    ratings.rename(columns={'movieId': 'itemId'}, inplace=True)

# Map userId and itemId to be zero-indexed
unique_users = ratings['userId'].unique()
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
ratings['userId'] = ratings['userId'].map(user_to_idx)

unique_items = ratings['itemId'].unique()
item_to_idx = {item: idx for idx, item in enumerate(unique_items)}
ratings['itemId'] = ratings['itemId'].map(item_to_idx)

# Now calculate num_users and num_items from the correctly prepared DataFrame
num_users = ratings["userId"].nunique()
num_items = ratings["itemId"].nunique()

print("Users:", num_users)
print("Items:", num_items)

ratings.head()

Users: 610
Items: 9724


,userId,itemId,rating,timestamp
0,0,0,4.0,964982703
1,0,1,4.0,964981247
2,0,2,4.0,964982224
3,0,3,5.0,964983815
4,0,4,5.0,964982931


In [33]:
# Initialize Model

model = LightGCN(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=config["model"]["embed_size"]
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["train"]["learning_rate"]
)

print(model)

LightGCN(
  (user_embedding): Embedding(610, 64)
  (item_embedding): Embedding(9724, 64)
)


In [34]:
# Prepare Training Data

users = torch.tensor(ratings["userId"].values)
items = torch.tensor(ratings["itemId"].values)

In [40]:
# Training Loop

epochs = config["train"]["epochs"]

for epoch in range(epochs):

    optimizer.zero_grad()

    user_emb, item_emb = model()

    scores = (user_emb[users] * item_emb[items]).sum(dim=1)

    loss = -torch.log(torch.sigmoid(scores)).mean()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

Epoch 0 Loss 3.2655
Epoch 5 Loss 3.2078
Epoch 10 Loss 3.1510
Epoch 15 Loss 3.0951
Epoch 20 Loss 3.0401
Epoch 25 Loss 2.9860
Epoch 30 Loss 2.9328
Epoch 35 Loss 2.8805
Epoch 40 Loss 2.8291
Epoch 45 Loss 2.7786
Epoch 50 Loss 2.7289
Epoch 55 Loss 2.6800
Epoch 60 Loss 2.6319
Epoch 65 Loss 2.5846
Epoch 70 Loss 2.5380
Epoch 75 Loss 2.4922
Epoch 80 Loss 2.4470
Epoch 85 Loss 2.4025
Epoch 90 Loss 2.3587
Epoch 95 Loss 2.3155
Epoch 100 Loss 2.2729
Epoch 105 Loss 2.2309
Epoch 110 Loss 2.1894
Epoch 115 Loss 2.1485
Epoch 120 Loss 2.1081
Epoch 125 Loss 2.0682
Epoch 130 Loss 2.0287
Epoch 135 Loss 1.9897
Epoch 140 Loss 1.9512
Epoch 145 Loss 1.9131
Epoch 150 Loss 1.8753
Epoch 155 Loss 1.8380
Epoch 160 Loss 1.8010
Epoch 165 Loss 1.7644
Epoch 170 Loss 1.7280
Epoch 175 Loss 1.6920
Epoch 180 Loss 1.6563
Epoch 185 Loss 1.6209
Epoch 190 Loss 1.5858
Epoch 195 Loss 1.5509
Epoch 200 Loss 1.5163
Epoch 205 Loss 1.4819
Epoch 210 Loss 1.4478
Epoch 215 Loss 1.4139
Epoch 220 Loss 1.3802
Epoch 225 Loss 1.3468
Epoch 230 

In [41]:
# Prepare Training Data

users = torch.tensor(ratings["userId"].values)
items = torch.tensor(ratings["itemId"].values)

In [43]:
# Generate Recommendations

user_id = 1

user_vector = model.user_embedding.weight[user_id]

scores = torch.matmul(
    model.item_embedding.weight,
    user_vector
)

topk = torch.topk(scores, config["train"]["top_k"])

print("Recommended items:", topk.indices)

Recommended items: tensor([3170, 4939, 1940, 1402,  724,  557, 4351, 6918,  295, 3998, 4773,  121,
         362, 3349, 7319, 3345, 2123, 2987, 3482, 3436])


In [46]:
# Evaluate Precision@K

recommended = topk.indices.numpy()
relevant = ratings[ratings["userId"] == user_id]["itemId"].values

precision = precision_at_k(recommended, relevant, config["train"]["top_k"])

print("Precision@K:", precision)

Precision@K: 0.0


In [47]:
# Save Model

MODEL_PATH = "/content/drive/MyDrive/lightgcn-res/results/lightgcn_model_movln_latest_small.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved")

Model saved
